In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from upath import UPath

import mbt

In [4]:
bin_file = UPath("../data/other/fov-2-scan-1.bin")

In [5]:
with mbt.MibiFile(file_path=bin_file, num_chunks=32) as f:
    panel = f.panel
    data = f.data

In [7]:
data.collect()

pixel_x,pixel_y,trigger_index,pulse_time,pulse_width,pulse_intensity
u32,u32,u16,u16,u8,u16
0,0,1,7237,2,1484
0,0,1,8132,3,4492
0,0,1,12392,2,2121
0,0,2,3127,2,2736
0,0,2,10747,2,4055
…,…,…,…,…,…
1023,1023,92,5544,4,10588
1023,1023,93,3025,4,7238
1023,1023,93,3035,3,7467


In [8]:
f.write_dataset_to_zarr(output_directory=UPath("test/"))

Storage Chunks: (1, 256, 256), Shard Shape: (39, 1024, 1024)


/Users/srivarra/Dev/Projects/Personal/mbt/.venv/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/srivarra/Dev/Projects/Personal/mbt/.venv/lib/python3.12/site-packages/zarr/core/array.py:4275: UserWarning: The dtype `<U14` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  meta = AsyncArray._create_metadata_v3(
/Users/srivarra/Dev/Projects/Personal/mbt/.venv/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/srivarra/Dev/Projects/Personal/mbt/.ve

In [9]:
f.write_ome_zarr(output_directory=UPath("test2/"), image_types=["counts"])

2025-07-10 12:43:23.940 | INFO     | mbt.io.zarr:write_ome_zarr:316 - OME-Zarr for 'counts' written to test2/2022-07-04_NBL_TMA2/fov-2-R2C1-counts.ome.zarr.
2025-07-10 12:43:23.940 | SUCCESS  | mbt.core.mibifile:write_ome_zarr:586 - Successfully wrote OME-Zarr to test2 for ../data/other/fov-2-scan-1.bin


In [10]:
import xarray as xr

In [ ]:
fov2_r2c1_ds: xr.Dataset = xr.open_dataset(
    UPath("./test/2022-07-04_NBL_TMA2/fov-2-R2C1.zarr"), engine="zarr", consolidated=True
)

fov2_r2c1_ds["intensity"]

/Users/srivarra/Dev/Projects/Personal/mbt/.venv/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


<xarray.DataArray 'intensity' (c: 39, y: 1024, x: 1024)> Size: 327MB
[40894464 values with dtype=int64]
Coordinates:
  * c        (c) object 312B 'B3GAT1' 'B7H3' 'CD3' ... 'TrkB' 'Vimentin' 'pS6'
  * x        (x) float64 8kB 0.0 0.3906 0.7812 1.172 ... 398.4 398.8 399.2 399.6
  * y        (y) float64 8kB 0.0 0.3906 0.7812 1.172 ... 398.4 398.8 399.2 399.6
Attributes:
    image_type:  intensity
    fov_name:    fov-2-R2C1
    run_name:    2022-07-04_NBL_TMA2

In [19]:
f.write_tifffile(image_types=["counts", "intensity", "intensity_width"], output_directory=UPath("test/"))

2025-07-10 12:49:33.900 | SUCCESS  | mbt.core.mibifile:write_tifffile:606 - Successfully wrote TIFF files to test for ../data/other/fov-2-scan-1.bin (types: ['counts', 'intensity', 'intensity_width'])


In [23]:
import ngff_zarr
from rich import print

fov_model = ngff_zarr.from_ngff_zarr(UPath("./test2/2022-07-04_NBL_TMA2/fov-2-R2C1-counts.ome.zarr"), validate=True)

print(fov_model)

Multiscales(
    images=[
        NgffImage(
            data=dask.array<from-zarr, shape=(39, 1024, 1024), dtype=uint32, chunksize=(1, 256, 256), 
chunktype=numpy.ndarray>,
            dims=['c', 'y', 'x'],
            scale={'c': 1, 'y': 0.390625, 'x': 0.390625},
            translation={'c': 0, 'y': 0, 'x': 0},
            name='image',
            axes_units={'c': None, 'y': 'micrometer', 'x': 'micrometer'},
            computed_callbacks=[]
        )
    ],
    metadata=Metadata(
        axes=[
            Axis(name='c', type='channel', unit=None),
            Axis(name='y', type='space', unit='micrometer'),
            Axis(name='x', type='space', unit='micrometer')
        ],
        datasets=[
            Dataset(
                path='fov-2-R2C1_counts',
                coordinateTransformations=[
                    Scale(scale=[1, 0.390625, 0.390625], type='scale'),
                    Translation(translation=[0, 0, 0], type='translation')
                ]
            )
        ],
        coordinateTransformations=[
            {'scale': [1, 0.390625, 0.390625], 'type': 'scale'},
            {'translation': [0, 0, 0], 'type': 'translation'}
        ],
        omero=None,
        name='image'
    ),
    scale_factors=None,
    method=None,
    chunks=None
)

In [25]:
xr.open_zarr(UPath("./test2/2022-07-04_NBL_TMA2/fov-2-R2C1-counts.ome.zarr"))

<xarray.Dataset> Size: 164MB
Dimensions:            (c: 39, y: 1024, x: 1024)
Dimensions without coordinates: c, y, x
Data variables:
    fov-2-R2C1_counts  (c, y, x) uint32 164MB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
Attributes:
    ome:      {'version': '0.5', 'multiscales': [{'axes': [{'name': 'c', 'typ...

In [26]:
fov_model

Multiscales(images=[NgffImage(data=dask.array<from-zarr, shape=(39, 1024, 1024), dtype=uint32, chunksize=(1, 256, 256), chunktype=numpy.ndarray>, dims=['c', 'y', 'x'], scale={'c': 1, 'y': 0.390625, 'x': 0.390625}, translation={'c': 0, 'y': 0, 'x': 0}, name='image', axes_units={'c': None, 'y': 'micrometer', 'x': 'micrometer'}, computed_callbacks=[])], metadata=Metadata(axes=[Axis(name='c', type='channel', unit=None), Axis(name='y', type='space', unit='micrometer'), Axis(name='x', type='space', unit='micrometer')], datasets=[Dataset(path='fov-2-R2C1_counts', coordinateTransformations=[Scale(scale=[1, 0.390625, 0.390625], type='scale'), Translation(translation=[0, 0, 0], type='translation')])], coordinateTransformations=[{'scale': [1, 0.390625, 0.390625], 'type': 'scale'}, {'translation': [0, 0, 0], 'type': 'translation'}], omero=None, name='image'), scale_factors=None, method=None, chunks=None)